# DIRE quick start — synthetic data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DIRE-OSI/platform/blob/main/notebooks/dire_quickstart_synthetic.ipynb)

This notebook walks through the **shape of the DIRE workflow** — monthly covariates and case counts per administrative unit, multi-horizon forecasting, and comparison against a simple baseline — using **fully synthetic data generated in the notebook**.

> **SYNTHETIC DATA.** Every number below is simulated. It is not surveillance data, not a DIRE forecast, and must not be used for any public-health decision. The model here is a lightweight stand-in for illustration; DIRE's methods are described in the [Methods & model card](https://dire-osi.github.io/platform/METHODS.html).

Runs in a few seconds with `numpy`, `pandas`, `scikit-learn` and `matplotlib` (all available on Colab).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

rng = np.random.default_rng(42)

## 1. Generate a synthetic panel

Twelve synthetic administrative units, eight years of monthly data. Incidence responds to lagged temperature and rainfall plus a seasonal cycle, with unit-level differences in baseline risk. Columns follow the [Data dictionary](https://dire-osi.github.io/platform/DATA_DICTIONARY.html).

In [ ]:
months = pd.period_range("2017-01", "2024-12", freq="M")
units = [f"SYN-{i:02d}" for i in range(1, 13)]
rows = []
for u in units:
    pop = rng.integers(50_000, 900_000)
    base = rng.uniform(5, 60)                     # baseline incidence per 100k
    phase = rng.uniform(0, 2 * np.pi)
    t_prev = r_prev = 0.0
    for m in months:
        season = np.sin(2 * np.pi * (m.month - 1) / 12 + phase)
        t2m = 24 + 3 * season + rng.normal(0, 0.8)
        precip = max(0, 120 + 90 * season + rng.normal(0, 30))
        dir_total = max(0, base * (1 + 0.9 * np.tanh(0.6 * t_prev) + 0.004 * r_prev) * (1 + 0.6 * season)
                        * rng.lognormal(0, 0.25))
        rows.append(dict(admin_id=u, year=m.year, month=m.month, period=m, pop_total=pop,
                         t2m_mean=t2m, precip_total=precip,
                         dir_total=dir_total, cases_total=rng.poisson(dir_total * pop / 1e5),
                         is_synthetic=True))
        t_prev, r_prev = t2m - 24, precip - 120
df = pd.DataFrame(rows)
df.head()

## 2. Build features and targets for three horizons

For each unit and issue month we use recent incidence and climate lags, and predict incidence for the **current month (h=0)**, **+1** and **+2** months — one model per horizon, as in DIRE.

In [ ]:
df = df.sort_values(["admin_id", "period"]).reset_index(drop=True)
g = df.groupby("admin_id")
for lag in (1, 2, 3, 12):
    df[f"dir_lag{lag}"] = g["dir_total"].shift(lag)
for lag in (1, 2):
    df[f"t2m_lag{lag}"] = g["t2m_mean"].shift(lag)
    df[f"precip_lag{lag}"] = g["precip_total"].shift(lag)
df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)
for h in (0, 1, 2):
    df[f"target_h{h}"] = g["dir_total"].shift(-h)

features = [c for c in df.columns if c.startswith(("dir_lag", "t2m_lag", "precip_lag", "month_"))]
data = df.dropna(subset=features + ["target_h0", "target_h1", "target_h2"])
train = data[data["period"] < pd.Period("2023-01", "M")]
test = data[data["period"] >= pd.Period("2023-01", "M")]
len(train), len(test)

## 3. Train, then compare with a calendar-month baseline

The baseline predicts each unit's historical average for the same calendar month — a standard reference forecast. Errors are reported as NRMSE (RMSE divided by the mean observed value).

In [ ]:
def nrmse(y, yhat):
    return np.sqrt(np.mean((y - yhat) ** 2)) / np.mean(y)

clim = train.groupby(["admin_id", "month"])["dir_total"].mean()
results, preds = [], {}
for h in (0, 1, 2):
    model = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=0)
    model.fit(train[features], train[f"target_h{h}"])
    yhat = model.predict(test[features])
    target_month = ((test["month"] - 1 + h) % 12) + 1
    base = clim.reindex(list(zip(test["admin_id"], target_month))).to_numpy()
    y = test[f"target_h{h}"].to_numpy()
    results.append(dict(horizon=h, model_nrmse=nrmse(y, yhat), baseline_nrmse=nrmse(y, base)))
    preds[h] = yhat
pd.DataFrame(results).round(3)

## 4. Plot one synthetic unit

In [ ]:
u = "SYN-03"
mask = (test["admin_id"] == u).to_numpy()
x = test.loc[mask, "period"].dt.to_timestamp()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(x, test.loc[mask, "target_h0"], color="#12263f", lw=2, label="Synthetic observed")
ax.plot(x, preds[0][mask], color="#1f5c99", ls="--", label="Model, h=0")
ax.set_ylabel("Incidence per 100,000")
ax.set_title(f"{u} — SYNTHETIC DATA, for illustration only")
ax.legend(frameon=False)
plt.tight_layout()

## Next steps

- Read the [Methods & model card](https://dire-osi.github.io/platform/METHODS.html) for how DIRE's production models differ from this illustration.
- Schema-faithful synthetic files for Peru and Brazil are described on the [Test data](https://dire-osi.github.io/platform/TEST_DATA.html) page.
- Questions or reproducibility reports: [Discussions](https://github.com/DIRE-OSI/platform/discussions).